In [7]:
import os, sys
import torch
import numpy as np
import pandas as pd 
import random
import json
import re

from pathlib import Path

# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo
import argparse


def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

def init_cfg(folder):
    cfg = Config.fromfile(f'{folder}/vfnet_r18.py')
    return cfg

def extract_first_band(path):
    # Regular expression to extract the first number after the first 'b'
    match = re.search(r'perfect_b(\d+)', path)

    # Extracted value
    if match:
        first_b_number = match.group(1)
        return int(first_b_number)

def list_weights(folder):
    return [x for x in Path(folder).rglob('*.pth') if 'best' not in str(x)]

def get_weight_b(band_sel):
    weigths = list_weights(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b{band_sel}')
    assert len(weigths) > 0, 'No weights found'
    weight_path = weigths[0]
    return weight_path


def set_NoiseTesting_config(weight_path, sensor: str, SNR: float, mtf_at_fe: float, Seed: int = 18):
    """
    Set the configuration for noise testing.
    Args:
        weight_path (str): The path to the weight file.
        sensor (str): The sensor: sentinel or venus
        SNR (float): The signal to noise ratio.
        mtf_at_fe (float): The modulation transfer function at the Nyquist frequency.
        Seed (int) : The seed for the random number generator.
    Returns:
        Config: The modified configuration object.
    """
    IMG_SIZE = 2048 # Default
    
    if isinstance(weight_path, str) :
        cfgDir = Path(weight_path).parent
    else:
        cfgDir = weight_path.parent
    
    
    cfg = init_cfg(folder=f'{cfgDir}')
    band_sel = 4
    assert band_sel is not None, f'Band not found in the weight path: {weight_path}'
    assert band_sel in [x for x in range(1,13,1)], f'Band not in the range [1,12]: {band_sel}'
    
    # Modify Config
    CHECKPOINT = weight_path
    cfg.load_from = f"{CHECKPOINT}"
    print(f'Loading from: {CHECKPOINT}')
    cfg.test_dataloader.dataset.pipeline = [{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': [band_sel]},
                        dict(type='LoadAnnotations', with_bbox=True),
                        dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                        dict(type='ImageCorruption', sensor=sensor, SNR=SNR, mtf_at_fe=mtf_at_fe), 
                        dict(
                            meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                    'height', 'width', 'instances', 'sample_idx', 
                                    'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                    'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                    'gt_bboxes_labels'),
                            type='PackDetInputs'),
                    ]

    work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
    cfg.work_dir = work_dir
    
    # Hook for using the SIoU intestead of the IoU:
    cfg.test_evaluator.type = 'SIoUCocoMetric'
    return cfg






SENSOR = 'sentinel'
    
    
    
Results = {'Band':[], 'SNR':[], 'MTF':[], 'Precision':[], 'Recall':[], 'F1':[]}

BAND = 10 if SENSOR == 'venus' else 8
mtf_tgt = 0.2 if SENSOR == 'venus' else 0.3

# SNR = 174 # Nominal SNR --> for S-2 at B8: 174 | for VENµS: 100. 
SNR_levels = np.geomspace(174, 1, 10)

MTFS = [0.2, 0.05, 0.001] if SENSOR == 'venus' else [0.3, 0.05, 0.001]

for mtf_tgt in MTFS:
    for SNR in SNR_levels:
        # TODO: Update weight path according to band_selected.
        weight_path = get_weight_b(BAND) if SENSOR == 'venus' else '/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Special/BS_4/LR_0.0005/IMG_2048/BANDS__b8/42_Optim_SGD/epoch_130.pth'
        cfg = set_NoiseTesting_config(weight_path, sensor=SENSOR, SNR=SNR, mtf_at_fe = mtf_tgt, Seed=18)

        # Run the test
        runner = RUNNERS.build(cfg)
        work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
        runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
        output_test_data = runner.test()
        
        ####### Save the results:
        P = output_test_data['coco/bbox_mAP']
        R = output_test_data['coco/bbox_AR@100']
        
        Results['Band'].append(BAND)
        Results['SNR'].append(SNR)
        Results['MTF'].append(mtf_tgt)
        
        Results['Precision'].append(P)
        Results['Recall'].append(R)
        try:
            F1 = 2 * (P * R) / (P + R)
        except ZeroDivisionError:
            F1 = 0
        Results['F1'].append(F1)
        

pd.DataFrame(Results).to_pickle(f'/Data_large/marine/PythonProjects/MMDET/studies/degradationStudy/{SENSOR}_SNR_study_b{BAND}.pkl')
print('Done')

Loading from: /Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Special/BS_4/LR_0.0005/IMG_2048/BANDS__b8/42_Optim_SGD/epoch_130.pth
09/14 18:33:13 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1608637542
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPAC

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


09/14 18:33:14 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
09/14 18:33:14 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


09/14 18:33:57 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
09/14 18:33:57 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

...

09/14 18:48:03 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
09/14 18:48:03 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


09/14 18:48:35 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
09/14 18:48:35 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


09/14 18:49:08 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
09/14 18:49:08 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i